# GaAs bands and DOS with py4vasp (vaspout.h5)

Run this notebook in `Common/examples/05-band-dos-plotting` after `job_band_dos.slurm` has finished.

Requirements
* VASP compiled with HDF5 support (`-DVASP_HDF5` in makefile.include), so that every run writes `vaspout.h5`
* `pip install py4vasp` (0.11.x at the time of writing); `pip install kaleido` to save static images
* `LORBIT = 11` in the band and DOS runs for projections

No output is stored in this notebook: the numbers you see depend on your own run.

In [ ]:
import py4vasp
print("py4vasp", py4vasp.__version__)

scf  = py4vasp.Calculation.from_path("scf")
band = py4vasp.Calculation.from_path("band")
dos  = py4vasp.Calculation.from_path("dos")

# Reference energy: E-fermi of the dense-mesh DOS run (the line-mode run does not give a reliable E_F)
ef = dos.dos.read()["fermi_energy"]
print("E_F (DOS run) =", ef, "eV")

## 1. Band structure along the line-mode path
`fermi_energy=` shifts the energies so that the given reference is at 0 eV.

In [ ]:
g = band.band.plot(fermi_energy=ef)
g.yrange = (-6, 4)
g

## 2. Fat bands
Selections use the py4vasp syntax: `"Ga(s), As(p)"`, atom indices `"1(d)"` (1-based), sums `"dxy + dxz + dyz"`.
For spin-polarised runs add `up` / `down`, e.g. `"up(As(p))"`; for SOC runs `sigma_x`, `sigma_z`.
Use `band.band.selections()` to list what is available.

In [ ]:
print(band.band.selections())
g = band.band.plot("Ga(s), As(p)", fermi_energy=ef, width=0.5)
g.yrange = (-6, 4)
g

## 3. Density of states with projections

In [ ]:
g = dos.dos.plot("Ga, As(s), As(p)")
g.xrange = (-8, 6)
g

## 4. Band gap (py4vasp reads the band-gap data only from VASP >= 6.5 output)

In [ ]:
print(scf.bandgap)

## 5. Export data and figures
`to_frame()` returns a pandas DataFrame; `to_csv()` writes it for Origin/Excel/matplotlib.
`to_image()` needs the kaleido package; prefer PDF/SVG for print.

In [ ]:
band.band.to_csv(filename="band_py4vasp.csv")
dos.dos.to_csv("Ga, As", filename="dos_py4vasp.csv")
df = dos.dos.to_frame("Ga, As")
print(df.head())
# band.band.to_image(filename="band_py4vasp.pdf")   # requires: pip install kaleido

## 6. One-step alternative: KPOINTS_OPT (VASP >= 6.3)
If the SCF run also had a `KPOINTS_OPT` file with the line-mode path, the bands along the path
are stored in the same `vaspout.h5` and can be plotted with the `"kpoints_opt"` selection:

```python
scf.band.plot("kpoints_opt")
```